## Initial steps

In [ ]:
import torch
import pandas as pd
import numpy as np
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import os
import time
import matplotlib.pyplot as plt
from sklearn import metrics
from torch.optim.lr_scheduler import LambdaLR
from tqdm import tqdm
from efficient_kan import KAN
import collections.abc as c
import import_ipynb

import MyFunctions, MyClassess


In [ ]:
##Loading functions from the auxiliary notebooks
%run MyFunctions.ipynb 
%run MyClassess.ipynb

In [ ]:
smarts=createSmarts()
func_group_names = pd.Series(smarts.keys())
func_group_names

In [ ]:
#Path to the directory containing the .csv files used to train the classifier for a given functional group. (Data from NIST database. The .csv files are generated from the .jdx files by a separate script, 
#detailed procedure of creation of .csv files (including scripts) described in article https://doi.org/10.1016/j.chemolab.2025.105421). 
# The FG3Dataset class (in the MyClasses file) is responsible for loading data
# from the ENFSI dataset. The paths to the directories are also specified there.
func_groups_path = os.path.join('D:\\SHAPE\\1VII\\functional-cnn-main\\ALL\\functional_groups')
func_groups_path

In [ ]:
#datasets loading and dataloaders creation (both NIST and ENFSI datasets)
func_groups_data, func_groups_datasets, func_groups_dataloaders = createHashmaps()
func_groups_dataN, func_groups_datasetsN, func_groups_dataloadersN = createHashmaps1()


func_groups_dataA, func_groups_datasetsA, func_groups_dataloadersA=createHashmapsCombined()

In [ ]:
#setting device (GPU or CPU)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
torch.cuda.is_available()
#device = torch.device('cpu')

In [ ]:
func_groups_dataN, func_groups_datasetsN, func_groups_dataloadersN = createHashmaps1()

## This section is used to train the models on spectra from the NIST database (pre-training).

In [ ]:
#The training function adapted to the NIST dataset uses the func_groups_dataloaders data loader;
#seeding is DISABLED in the code (the seed argument is retained in the method signature for backward compatibility with our other functions, although it is not used..

def train(num_epochs, lossFn, optimizer, group: str, weight_decay, learning_rate, kan_layers, lambda_lr, seed=42, save: bool = False, plot: bool = False, disable_verbose: bool = False, iter=0, dirpath=""):
    train_losses, test_losses = [], []


    model = ChemicyBad(kan_layers).to(device)

    #torch.manual_seed(seed)
    
    optimizer = optimizer(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = LambdaLR(optimizer, lr_lambda=lambda epoch: lambda_lr ** epoch)
    #scheduler = ExponentialLR(optimizer, gamma=0.9)
    global descr;
    for epoch in range(num_epochs):
        
        model.train()
        train_loss = 0.
        with tqdm(func_groups_dataloaders[group]['training'], disable=disable_verbose) as pbar:
            f1 = 0
            lista_grup = np.zeros(1)
            list_func = np.zeros(1)
            for y_batch, func_group_batch in pbar:
                #print(y_batch)
                y_batch, func_group_batch = y_batch.to(device), func_group_batch.to(device)
                # print(y_batch.shape)
                # print(y_batch[0])
                
                optimizer.zero_grad()
                output = model(y_batch).squeeze()
                
                loss = lossFn(output, func_group_batch)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

                for i in range(output.shape[0]):
                    #F1 metryka
                    predicted_well = ((torch.round(output[i]) == func_group_batch[i]) * 1 * func_group_batch[i]).item()

                    lista_grup += predicted_well
                    list_func += func_group_batch[i].item()

                f1 = metrics.f1_score(torch.Tensor.cpu(func_group_batch).detach().numpy(), torch.Tensor.cpu(torch.round(output)).detach().numpy())
                pbar.set_postfix(epoch=epoch, loss=train_loss, accuracy=(lista_grup/list_func).item(), f1=f1, lr=optimizer.param_groups[0]['lr'])

                train_acc = (lista_grup/list_func).item()
            train_loss /= len(func_groups_dataloaders[group]['training'])
            train_losses.append(train_loss)
         

        model.eval()
        test_loss = 0.

        with torch.no_grad():
            with tqdm(func_groups_dataloaders[group]['test'], disable=disable_verbose) as tbar:
                f1 = 0
                lista_grup = np.zeros(1)
                list_func = np.zeros(1)
                for y_batch, func_group_batch in tbar:
                    y_batch, func_group_batch = y_batch.to(device), func_group_batch.to(device)

                    output = model(y_batch).squeeze()
                    loss = lossFn(output, func_group_batch)
                    test_loss += loss.item()

                    for i in range(output.shape[0]):
                        predicted_well = ((torch.round(output[i]) == func_group_batch[i]) * 1 * func_group_batch[i]).item()

                        lista_grup += predicted_well
                        list_func += func_group_batch[i].item()
                    f1 = metrics.f1_score(torch.Tensor.cpu(func_group_batch).detach().numpy(), torch.Tensor.cpu(torch.round(output)).detach().numpy())
                    tbar.set_postfix(epoch=epoch, loss=test_loss, accuracy=(lista_grup/list_func).item(), f1=f1)

                    test_acc = (lista_grup/list_func).item()
                test_loss /= len(func_groups_dataloaders[group]['test'])
                test_losses.append(test_loss)
        descrtemp='epoch '+ str(epoch)+ " tr_loss "+ str(train_loss)+ " test loss "+ str(test_loss);
        descr+=descrtemp+"\n";
        print(descrtemp)
        scheduler.step()

        if epoch%5==0:
            torch.save(model, os.path.join(dirpath, group+"_"+str(iter)+"_"+str(epoch)+".pt"))
        


    if plot:
        plt.plot(train_losses, label='training loss')
        plt.plot(test_losses, label='test loss')
        plt.legend()
        plt.title(f'{group}')
        plt.plot()
        plt.savefig(dirpath+"\\Learning_progress_"+str(iter)+".png")
        plt.show()

    model.tr_accuracy = np.round(train_acc,2)
    model.te_accuracy = np.round(test_acc,2)
    model.tr_loss = np.round(train_losses[-1],2)
    model.te_loss = np.round(test_losses[-1],2)

    return model

In [ ]:
#Repeated training of neural networks for user-specified functional groups. The trained models are saved in the specified directory.
save_dirpath= "C:\\Users\\Tomek_\\Desktop\\cc\\TrainedModelsCNNKANFinalLR095_VIII2026_ALLGR"
save_part_path="C:\\Users\\Tomek_\\Desktop\\cc\\TrainedModelsCNNKANFinalLR095_VIII2026_ALLGR\partial"

#groups=['alkyl halides', 'amides', 'amines', 'aryl chlorides', 'carboxylic acids', 'carboxylic esters',  'ether',  'heterocyclic',  'nitriles',  'phenols']
groups=['alcohols', 'alkane', 'alkyl aryl ethers']

for group1 in groups:
    crossEnt = torch.nn.BCELoss()
    epochNum=25;    
    numberOfTrials=12
    summary_description=""
    descr=""
    for i in range(numberOfTrials):   
        descr+="Trial "+str(i)+"\n";
        func_groups_data, func_groups_datasets, func_groups_dataloaders = createHashmaps()#tworzymy od nowa zestaw testowy
    
        model = train(num_epochs=epochNum, lossFn=crossEnt, 
                                    optimizer=torch.optim.AdamW,
                                    group = group1 ,
                                    weight_decay=1e-2,
                                    learning_rate=5e-3,
                                    kan_layers=[10,5,1],
                                    lambda_lr=0.95,
                                    save=False,
                                    plot=True,
                                    disable_verbose=True,iter=i,dirpath=save_part_path)
    
    
        torch.save(model, os.path.join(save_dirpath, f'{group1}_{i}.pt'))
        training_acc(model, group1,i)
        test_acc(model, group1,i)
        
    desc_file_path=save_dirpath+"\\"+group1+"_NIST_testAcc.txt."
    desc_file_path_training=save_dirpath+"\\"+group1+"_NIST_trainDescr.dat"
    with open(desc_file_path, "w", encoding="utf-8") as f:
         f.write(summary_description) 
    with open(desc_file_path_training, "w", encoding="utf-8") as f:
         f.write(descr) 


# Repeated training from scratch on a new dataset derived from the ENFSI database (this is not transfer learning yet, but our reference method).

In [ ]:
#Repeated training of neural networks for user-specified functional group. The trained models are saved in the specified directory.
save_dirpath= "C:\\Users\\Tomek_\\Desktop\\cc\\TrainedModelsCNNKANFinalLR095_VIII2026_nowyZbiórOdZera"
save_part_path="C:\\Users\\Tomek_\\Desktop\\cc\\TrainedModelsCNNKANFinalLR095_VIII2026_nowyZbiórOdZera\partial"
crossEnt = torch.nn.BCELoss()
epochNum=25;

group1="ketones"
#group1="methyl"
#group1="esters";
#group1="alkene"
#group1="aromatics"
#group1="nitro"
#group1="nitriles"
numberOfTrials=12
summary_description=""
descr=""
for i in range(numberOfTrials):   
    descr+="Trial "+str(i)+"\n";
    #Note: For the aromatics group, the split ratio should be set to 0.26–0.74, because with a 0.25 split, a single spectrum unfortunately remains in the last training batch.
    func_groups_data, func_groups_datasets, func_groups_dataloaders = createHashmaps1(test_ratio=0.25)#tworzymy od nowa zestaw testowy

    model = train(num_epochs=epochNum, lossFn=crossEnt, 
                                optimizer=torch.optim.AdamW,
                                group = group1 ,
                                weight_decay=1e-2,
                                learning_rate=5e-3,
                                kan_layers=[10,5,1],
                                lambda_lr=0.95,
                                save=False,
                                plot=True,
                                disable_verbose=True,iter=i,dirpath=save_part_path)


    torch.save(model, os.path.join(save_dirpath, f'{group1}_{i}.pt'))
    training_acc(model, group1,i)
    test_acc(model, group1,i)
    
desc_file_path=save_dirpath+"\\"+group1+"_ENFSI_testAcc.txt."
desc_file_path_training=save_dirpath+"\\"+group1+"_ENFSI_trainDescr.dat"
with open(desc_file_path, "w", encoding="utf-8") as f:
     f.write(summary_description) 
with open(desc_file_path_training, "w", encoding="utf-8") as f:
     f.write(descr) 


# Fine-tuning on the ENFSI dataset with selected layers frozen



In [ ]:
#Defining a function for fine-tuning an existing model on a new dataset using func_groups_dataloadersN

def retrain(currentModel, num_epochs, lossFn, optimizer, group: str, weight_decay, learning_rate, kan_layers, lambda_lr, seed=42, save: bool = False, plot: bool = False, disable_verbose: bool = False, iter=0, dirpath=""):
    train_losses, test_losses = [], []
    global descr;

    model = currentModel.to(device)

    #torch.manual_seed(seed)

    optimizer = optimizer(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = LambdaLR(optimizer, lr_lambda=lambda epoch: lambda_lr ** epoch)
    #scheduler = ExponentialLR(optimizer, gamma=0.9)

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.
        with tqdm(func_groups_dataloadersN[group]['training'], disable=disable_verbose) as pbar:
            f1 = 0
            lista_grup = np.zeros(1)
            list_func = np.zeros(1)
            for y_batch, func_group_batch in pbar:
                y_batch, func_group_batch = y_batch.to(device), func_group_batch.to(device)

                optimizer.zero_grad()
                output = model(y_batch).squeeze()
                loss = lossFn(output, func_group_batch)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

                for i in range(output.shape[0]):
                    #F1 metryka
                    predicted_well = ((torch.round(output[i]) == func_group_batch[i]) * 1 * func_group_batch[i]).item()

                    lista_grup += predicted_well
                    list_func += func_group_batch[i].item()

                f1 = metrics.f1_score(torch.Tensor.cpu(func_group_batch).detach().numpy(), torch.Tensor.cpu(torch.round(output)).detach().numpy())
                pbar.set_postfix(epoch=epoch, loss=train_loss, accuracy=(lista_grup/list_func).item(), f1=f1, lr=optimizer.param_groups[0]['lr'])

                train_acc = (lista_grup/list_func).item()
            train_loss /= len(func_groups_dataloadersN[group]['training'])
            train_losses.append(train_loss)
        


        model.eval()
        test_loss = 0.

        with torch.no_grad():
            with tqdm(func_groups_dataloadersN[group]['test'], disable=disable_verbose) as tbar:
                f1 = 0
                lista_grup = np.zeros(1)
                list_func = np.zeros(1)
                for y_batch, func_group_batch in tbar:
                    y_batch, func_group_batch = y_batch.to(device), func_group_batch.to(device)

                    output = model(y_batch).squeeze()
                    loss = lossFn(output, func_group_batch)
                    test_loss += loss.item()

                    for i in range(output.shape[0]):
                        predicted_well = ((torch.round(output[i]) == func_group_batch[i]) * 1 * func_group_batch[i]).item()

                        lista_grup += predicted_well
                        list_func += func_group_batch[i].item()
                    f1 = metrics.f1_score(torch.Tensor.cpu(func_group_batch).detach().numpy(), torch.Tensor.cpu(torch.round(output)).detach().numpy())
                    tbar.set_postfix(epoch=epoch, loss=test_loss, accuracy=(lista_grup/list_func).item(), f1=f1)

                    test_acc = (lista_grup/list_func).item()
                test_loss /= len(func_groups_dataloadersN[group]['test'])
                test_losses.append(test_loss)
                
        descrtemp='epoch '+ str(epoch)+ " tr_loss "+ str(train_loss)+ " test loss "+ str(test_loss);
        descr+=descrtemp+"\n";
        print(descrtemp)
        scheduler.step()
        
        if epoch%5==0:
            torch.save(model, os.path.join(dirpath, group+"_"+str(iter)+"_"+str(epoch)+".pt"))
        

    #if save:
    #    torch.save(model, os.path.join('..','ALL', 'trained_models', group, f'{group}_{time.strftime(r"%m:%d:%Y, %H:%M:%S")}.pt'))

    if plot:
        plt.plot(train_losses, label='training loss')
        plt.plot(test_losses, label='test loss')
        plt.legend()
        plt.title(f'{group}')
        plt.plot()
        plt.savefig(dirpath+"\\Learning_progress_"+str(iter)+".png")
        plt.show()
        opis=str(train_losses)+"\n"+str(test_losses)
    global training_descr
    training_descr+=opis;
    model.tr_accuracy = np.round(train_acc,2)
    model.te_accuracy = np.round(test_acc,2)
    model.tr_loss = np.round(train_losses[-1],2)
    model.te_loss = np.round(test_losses[-1],2)

    return model

In [ ]:
#For the specified functional groups, the pre-trained models are loaded from the path provided in the path variable and fine-tuned on the ENFSI dataset.
#The models are saved in the specified directory, along with training statistics and model checkpoints saved every 5 epochs. 
#This cell also displays the model weights to allow confirmation and verification that the weights of the initial layers are indeed frozen.

save_dirpath= "C:\\Users\\Tomek_\\Desktop\\cc\\TrainedModelsCNNKANFinalLR095_VIII2026_nowyRetrainFrozen\\"
save_part_path= "C:\\Users\\Tomek_\\Desktop\\cc\\TrainedModelsCNNKANFinalLR095_VIII2026_nowyRetrainFrozen\\partial\\"
path="C:\\Users\\Tomek_\\Desktop\\cc\\TrainedModelsCNNKANFinalLR095_VIII20261\\"

groups=["ketones","methyl","esters","alkene","aromatics","nitro"]

crossEnt = torch.nn.BCELoss()
epochNum=25;

for group1 in groups:
    training_descr=""
    summary_description=""
    descr=""  
    for i in range(12):    
        modelOLD= torch.load(path+group1+"_"+str(i)+".pt", weights_only=False)
        #for name, p in model.named_parameters():
        #    print(p)
        
        descr+="Proba "+str(i)+"\n";
        func_groups_data, func_groups_datasets, func_groups_dataloaders = createHashmaps1(test_ratio=0.25)#tworzymy od nowa zestaw testowy
        #frozing of selected layers - very important step!!!!
        for name, p in modelOLD.named_parameters():
            print(name)            
            if name=="conv_1.weight" or name=="conv_1.bias" or name=="conv_2.weight" or name=="conv_2.bias" or name=="conv_3.weight" or name=="conv_3.bias":
                
                print("tu ",p.requires_grad)
                p.requires_grad=False
                print("tu ",p.requires_grad)
                
        model = retrain(currentModel=modelOLD,num_epochs=epochNum, lossFn=crossEnt, 
                                    optimizer=torch.optim.AdamW,
                                    group = group1 ,
                                    weight_decay=1e-2,
                                    learning_rate=2.5e-3,
                                    kan_layers=[0],
                                    lambda_lr=0.95,
                                    save=False,
                                    plot=True,
                                    disable_verbose=True,iter=i,dirpath=save_part_path)
    
    
        torch.save(model, os.path.join(save_dirpath, f'{group1}_{i}.pt'))
        training_acc(model, group1,i)
        test_acc(model, group1,i)
        
    desc_file_path=save_dirpath+"\\"+group1+"_ENFSI_retrain_testAcc.txt."
    desc_file_path_training=save_dirpath+"\\"+group1+"_ENFSI_retrain_trainDescr.dat"
    with open(desc_file_path, "w", encoding="utf-8") as f:
         f.write(summary_description) 
    with open(desc_file_path_training, "w", encoding="utf-8") as f:
         f.write(descr) 
    #for name, p in modelOLD.named_parameters():
    #        print(name)
    #        print(p)
    #for name, p in model.named_parameters():
    #        print(name)
    #        print(p)

# Fine-tuning on the ENFSI dataset

In [ ]:
#New wersion of retrain function, now all model weights allowed to be updated during training.

def retrain(currentModel, num_epochs, lossFn, optimizer, group: str, weight_decay, learning_rate, kan_layers, lambda_lr, seed=42, save: bool = False, plot: bool = False, disable_verbose: bool = False, iter=0, dirpath=""):
    train_losses, test_losses = [], []
    global descr;

    model = currentModel.to(device)

    #torch.manual_seed(seed)

    optimizer = optimizer(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = LambdaLR(optimizer, lr_lambda=lambda epoch: lambda_lr ** epoch)
    #scheduler = ExponentialLR(optimizer, gamma=0.9)

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.
        with tqdm(func_groups_dataloadersN[group]['training'], disable=disable_verbose) as pbar:
            f1 = 0
            lista_grup = np.zeros(1)
            list_func = np.zeros(1)
            for y_batch, func_group_batch in pbar:
                y_batch, func_group_batch = y_batch.to(device), func_group_batch.to(device)

                optimizer.zero_grad()
                output = model(y_batch).squeeze()
                loss = lossFn(output, func_group_batch)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

                for i in range(output.shape[0]):
                    #F1 metryka
                    predicted_well = ((torch.round(output[i]) == func_group_batch[i]) * 1 * func_group_batch[i]).item()

                    lista_grup += predicted_well
                    list_func += func_group_batch[i].item()

                f1 = metrics.f1_score(torch.Tensor.cpu(func_group_batch).detach().numpy(), torch.Tensor.cpu(torch.round(output)).detach().numpy())
                pbar.set_postfix(epoch=epoch, loss=train_loss, accuracy=(lista_grup/list_func).item(), f1=f1, lr=optimizer.param_groups[0]['lr'])

                train_acc = (lista_grup/list_func).item()
            train_loss /= len(func_groups_dataloadersN[group]['training'])
            train_losses.append(train_loss)
        


        model.eval()
        test_loss = 0.

        with torch.no_grad():
            with tqdm(func_groups_dataloadersN[group]['test'], disable=disable_verbose) as tbar:
                f1 = 0
                lista_grup = np.zeros(1)
                list_func = np.zeros(1)
                for y_batch, func_group_batch in tbar:
                    y_batch, func_group_batch = y_batch.to(device), func_group_batch.to(device)

                    output = model(y_batch).squeeze()
                    loss = lossFn(output, func_group_batch)
                    test_loss += loss.item()

                    for i in range(output.shape[0]):
                        predicted_well = ((torch.round(output[i]) == func_group_batch[i]) * 1 * func_group_batch[i]).item()

                        lista_grup += predicted_well
                        list_func += func_group_batch[i].item()
                    f1 = metrics.f1_score(torch.Tensor.cpu(func_group_batch).detach().numpy(), torch.Tensor.cpu(torch.round(output)).detach().numpy())
                    tbar.set_postfix(epoch=epoch, loss=test_loss, accuracy=(lista_grup/list_func).item(), f1=f1)

                    test_acc = (lista_grup/list_func).item()
                test_loss /= len(func_groups_dataloadersN[group]['test'])
                test_losses.append(test_loss)
                
        descrtemp='epoch '+ str(epoch)+ " tr_loss "+ str(train_loss)+ " test loss "+ str(test_loss);
        descr+=descrtemp+"\n";
        print(descrtemp)
        scheduler.step()
        
        if epoch%5==0:
            torch.save(model, os.path.join(dirpath, group+"_"+str(iter)+"_"+str(epoch)+".pt"))
        

    #if save:
    #    torch.save(model, os.path.join('..','ALL', 'trained_models', group, f'{group}_{time.strftime(r"%m:%d:%Y, %H:%M:%S")}.pt'))

    if plot:
        plt.plot(train_losses, label='training loss')
        plt.plot(test_losses, label='test loss')
        plt.legend()
        plt.title(f'{group}')
        plt.plot()
        plt.savefig(dirpath+"\\Learning_progress_"+str(iter)+".png")
        plt.show()
        opis=str(train_losses)+"\n"+str(test_losses)
    global training_descr
    training_descr+=opis;
    model.tr_accuracy = np.round(train_acc,2)
    model.te_accuracy = np.round(test_acc,2)
    model.tr_loss = np.round(train_losses[-1],2)
    model.te_loss = np.round(test_losses[-1],2)

    return model

In [ ]:
#as in previous section, but now all weights can be updated

save_dirpath= "C:\\Users\\Tomek_\\Desktop\\cc\\TrainedModelsCNNKANFinalLR095_VIII2026_nowyRetrainALLGR\\"
save_part_path= "C:\\Users\\Tomek_\\Desktop\\cc\\TrainedModelsCNNKANFinalLR095_VIII2026_nowyRetrainALLGR\\partial\\"
path="C:\\Users\\Tomek_\\Desktop\\cc\\TrainedModelsCNNKANFinalLR095_VIII2026_ALLGR\\"

#groups=["ketones","methyl","esters","alkene","aromatics","nitro"]
#groups=['alkyl halides', 'amides', 'amines', 'aryl chlorides', 'carboxylic acids', 'carboxylic esters',  'ether',  'heterocyclic',  'nitriles',  'phenols']
groups=['alcohols', 'alkane', 'alkyl aryl ethers']



crossEnt = torch.nn.BCELoss()
epochNum=25;

for group1 in groups:
    training_descr=""
    summary_description=""
    descr=""  
    for i in range(12):    
        modelOLD= torch.load(path+group1+"_"+str(i)+".pt", weights_only=False)
        descr+="Trial "+str(i)+"\n";
        func_groups_data, func_groups_datasets, func_groups_dataloaders = createHashmaps1(test_ratio=0.25)#tworzymy od nowa zestaw testowy
    
        model = retrain(currentModel=modelOLD,num_epochs=epochNum, lossFn=crossEnt, 
                                    optimizer=torch.optim.AdamW,
                                    group = group1 ,
                                    weight_decay=1e-2,
                                    learning_rate=2.5e-3,
                                    kan_layers=[0],
                                    lambda_lr=0.95,
                                    save=False,
                                    plot=True,
                                    disable_verbose=True,iter=i,dirpath=save_part_path)
    
    
        torch.save(model, os.path.join(save_dirpath, f'{group1}_{i}.pt'))
        training_acc(model, group1,i)
        test_acc(model, group1,i)
        
    desc_file_path=save_dirpath+"\\"+group1+"_ENFSI_retrain_testAcc.txt."
    desc_file_path_training=save_dirpath+"\\"+group1+"_ENFSI_retrain_trainDescr.dat"
    with open(desc_file_path, "w", encoding="utf-8") as f:
         f.write(summary_description) 
    with open(desc_file_path_training, "w", encoding="utf-8") as f:
         f.write(descr) 
    

In [ ]:
#single group manual retraining 
save_dirpath= "C:\\Users\\Tomek_\\Desktop\\cc\\TrainedModelsCNNKANFinalLR095_VIII2026_nowyRetrain1\\"
save_part_path= "C:\\Users\\Tomek_\\Desktop\\cc\\TrainedModelsCNNKANFinalLR095_VIII2026_nowyRetrain1\\partial\\"
path="C:\\Users\\Tomek_\\Desktop\\cc\\TrainedModelsCNNKANFinalLR095_VIII20261\\"

crossEnt = torch.nn.BCELoss()
epochNum=25;

#group1="ketones"
#group1="methyl"
#group1="esters";
group1="alkene"
#group1="aromatics"
#group1="nitro"
#group1="nitriles"

training_descr=""
summary_description=""
descr=""

for i in range(12):    
    modelOLD= torch.load(path+group1+"_"+str(i)+".pt", weights_only=False)
    descr+="Proba "+str(i)+"\n";
    func_groups_data, func_groups_datasets, func_groups_dataloaders = createHashmaps1(test_ratio=0.25)#tworzymy od nowa zestaw testowy

    model = retrain(currentModel=modelOLD,num_epochs=epochNum, lossFn=crossEnt, 
                                optimizer=torch.optim.AdamW,
                                group = group1 ,
                                weight_decay=1e-2,
                                learning_rate=2.55e-3,
                                kan_layers=[0],
                                lambda_lr=0.95,
                                save=False,
                                plot=True,
                                disable_verbose=True,iter=i,dirpath=save_part_path)


    torch.save(model, os.path.join(save_dirpath, f'{group1}_{i}.pt'))
    training_acc(model, group1,i)
    test_acc(model, group1,i)
    
desc_file_path=save_dirpath+"\\"+group1+"_ENFSI_retrain_testAcc.txt."
desc_file_path_training=save_dirpath+"\\"+group1+"_ENFSI_retrain_trainDescr.dat"
with open(desc_file_path, "w", encoding="utf-8") as f:
     f.write(summary_description) 
with open(desc_file_path_training, "w", encoding="utf-8") as f:
     f.write(descr) 
    